# `xdggs-dggrid4py`: A xdggs plugin for IGEO7 DGGS

`xdggs-dggrid4py` aims to provide IGEO7 DGGS indexing for xarray datasets. It consists of 2 main functions:

1. Easy access to IGEO7 indexed xarray datasets, including query the datasets with points in WGS84, or zone ID.
2. Convert raster dataset in GeoTiff format to IGEO7 indexed xarray datasets

## Converting a GeoTiff into an IGEO7 DGGS indexed xarray dataset using `xdggs-dggrid4py`
There are many ways to convert a GeoTiff into a DGGS indexed dataset. The main idea is to assign a pixel with a DGGS zone ID.
- Nearest neighbourhood: Assign the pixel to the nearest zone centroid, measured by the CRS supported by the DGGS. For IGEO7 DGGS, it uses WGS84. 
- Zonal statistics: Assign pixels that are within a zone of the DGGS. 
- Weighted average: Overlay the DGGS zone's grid on the raster grid, then, for each zone, calculate the percentage of area overlapped with pixels as the weights. Finally, calculate the zone's value using the weights and raster value.


This notebook demonstrates conversion using the nearest neighbourhood method with IGEO7 DGGS.

### General steps for conversion using the nearest-neighbourhood method. 

1. The first step is to determine the refinement level to use. It affects the number of rows of the dataset after conversion.
   - For example, if the refinement level approximately matches the spatial resoultion (in square meters), then the expected number of rows in the resulting dataset should be roughly equal to the number of zones of the region being converted. But if a coarser refinement level is used, the number of rows in the converted result should equal the number of pixels. 

2. Then, generate the zone centroids together with ID that exist in the region of the GeoTiff

3. Assign each pixel to the nearest zone by measuring the distance between the pixel and zone centroids.

For `xdggs-dggrid4py`, it used S2PointIndex from [pys2Index](https://github.com/benbovy/pys2index) to calculate the distance between pixel coordinates and the zone centroids in WGS84. 

### Demostration 

#### Install the package from `pypi` or install it from GitHub 

In [ ]:
# install it from pypi : 
# !pip install xdggs-dggrid4py
# or install it from GitHub for the lastest commits.
# pip install git+https://github.com/LandscapeGeoinformatics/xdggs-dggrid4py.git

#### Import libraries

In [1]:
import xarray as xr
import rioxarray as rio
import os
import warnings
warnings.filterwarnings("ignore")

# xdggs-dggrid4py require both dggrid4py and DGGRID installed
# export the path of DGGRID binary with the environment variable "DGGRID_PATH"
os.environ['DGGRID_PATH']= "/home/dick/micromamba/envs/xdggs/bin/dggrid"

#### Using rioxarray to load the GeoTiff into xarray dataset

In this demonstration, we use a sample collection prepared by the [Landscape and Geoinformatics Lab](https://landscape-geoinformatics.ut.ee/) of the University of Tartu, which is available on Zenodo.

The collection consists of three GeoTiff files (DEM, TWI, Slope). [![DOI](https://zenodo.org/badge/DOI/10.5281/zenodo.18877265.svg)](https://doi.org/10.5281/zenodo.18877265)


In [3]:
# Using DEM sample to demonstrate the conversion, you can also try with others sample using the following links
# Slope: https://zenodo.org/records/18877265/files/est_topo_slope_10m_clipped.tif?download=1
# TWI: https://zenodo.org/records/18877265/files/est_topo_twi_10m_clipped.tif?download=1

tiff_path = "https://zenodo.org/records/18877265/files/est_topo_dem_10m_clipped.tif?download=1"
raster_ds = rio.open_rasterio(tiff_path, band_as_variable=True, masked=True, chunks={'x':200, 'y':200})
raster_ds

<xarray.Dataset> Size: 8MB
Dimensions:      (x: 1779, y: 1081)
Coordinates:
  * x            (x) float64 14kB 6.334e+05 6.335e+05 ... 6.512e+05 6.512e+05
  * y            (y) float64 9kB 6.468e+06 6.468e+06 ... 6.457e+06 6.457e+06
    spatial_ref  int64 8B 0
Data variables:
    band_1       (y, x) float32 8MB dask.array<chunksize=(200, 200), meta=np.ndarray>
Attributes:
    AREA_OR_POINT:  Area

### Using xdggs-dggrid4py regridding module for conversion

#### Choose the conversion method
- As introduced at the beginning, there are several ways to perform the conversion.

  So xdggs-dggrid4py separates methods from regridding. Inside the `xdggs_dggrid4py.regridding.methods` module:
   - `centroid_based`: zone centroids based methods
      - `nearestpoint`: nearest point method implementation
      - `mapblocks_nearestpoint`: nearest point method implementation for parallelism
   - `zonal_based`: (to be implemented)
   - `weighted_average`: (to be implemented)

#### Running the regridding in parallel or just as a single process
- `xdggs-dggrid4py` supports conversion with parallelism or just a regular single process.
   - `xdggs_dggrid4py.regridding`
      - `regridding`: perform regridding as a single process  
      - `mapblocks_regridding`: perform regridding with parallelism
    
- To run conversion in parallel, it first partitions the GeoTiff into smaller blocks, then processes each block individually, as described in the general steps. Then, it combines the results from each block to form the final dataset to return. The parallelism is achieved using `dask.array.map_blocks`.
- Input parameters for regridding:
  - `ds`: the xarray dataset to be converted
  - `grid_name`: the name of DGGS to be used for conversion. `IGEO7`
  - `method`: method that used for regridding
  - `coordinates`: the coordinate variables name of the ds in list. Default to `[x, y]`
  - `original_crs`: CRS in wkt format. xdggs-dggrid4py will use the CRS from `crs_wkt` if it exist in the spatial_ref of `ds`. Otherwise, using the wkt supplied. Defualt to `None`. 
  - `refinement_level`: An integer to indicate at which refinement level to convert. Default to `-1`
     - If set to `-1`. It automatically calculates the finest refinement level that matches the spatial resolution. It is done by calculating the surface area of a sphere that represents the GeoTiff region, then dividing that area by the number of pixels to obtain the average area per pixel. The average area per pixel is used to determine the refinement level to be used.
  - `wgs84_geodetic_conversion`: Default to `True`.
       - convert the input clip bounds from WGS84 geodetic coordinates to the authalic sphere coordinates.
       - convert hexagon centroids from the authalic sphere to WGS84 geodetic before assigning pixels to zone centroids.
  - `dggs_vert0_lon`: The longitude of the initial vertex. It is set to 11.20 by default; users can change it to other values by passing the parameter to the function. Default to `11.20`
  - `zone_id_repr`: Which representation is used for the zone ID, it supports : `['int', 'hexstring', 'textual']`, default to 'int'.

In [4]:
%%time
# To perform conversion using nearest neighbourhood mehtod with parallelism using mapblocks
# import the mapblocks version of nearestpoint method 
from xdggs_dggrid4py.regridding.methods.centroid_based import mapblocks_nearestpoint
# import the mapblocks version of regridding
from xdggs_dggrid4py.regridding import mapblocks_regridding

igeo7_indexed_ds = mapblocks_regridding(ds=raster_ds,
                                      grid_name='igeo7', 
                                      method='mapblocks_nearestpoint',
                                      refinement_level=-1, 
                                      wgs84_geodetic_conversion=True, 
                                      zone_id_repr='int')


# To perform conversion using nearest neighbourhood mehtod as a single process
#from xdggs_dggrid4py.regridding.methods.centroid_based import nearestpoint
#from xdggs_dggrid4py.regridding import regridding
#igeo7_indexed_ds = regridding(ds=raster_ds,
#                              grid_name='IGEO7', 
#                              method='nearestpoint',
#                              refinement_level=-1, 
#                              wgs84_geodetic_conversion=True, 
#                              zone_id_repr='int')

Registered regridding method centerpoint
Registered regridding method nearestpoint
Registered regridding method mapblocks_nearestpoint
xdggs_dggrid4py.utils area of extent (km^2): 206.60011408860592
xdggs_dggrid4py.utils average area per square grid (km^2): 0.00010743082602019237
xdggs_dggrid4py.regridding.regridding auto refinement level: 14,                        estimate zones       : 2747342,                       estimate block size  : 101754
[########################################] | 100% Completed | 105.83 s
Registered regridding method centerpoint
Registered regridding method nearestpoint
Registered regridding method mapblocks_nearestpoint
Registered regridding method centerpoint
Registered regridding method nearestpoint
Registered regridding method mapblocks_nearestpoint
Registered regridding method centerpoint
Registered regridding method nearestpoint
Registered regridding method mapblocks_nearestpoint
Registered regridding method centerpoint
Registered regridding method n

Now, the 2D dataset is converted into a 1D dataset indexed by IGEO7 DGGRS. The related grid info is stored as attributes of the `zone_id` coordinate. 

In [5]:
igeo7_indexed_ds

<xarray.Dataset> Size: 33MB
Dimensions:      (zone_id: 2728520)
Coordinates:
  * zone_id      (zone_id) uint64 22MB 18667837283631103 ... 18805349850808319
    spatial_ref  int64 8B 0
Data variables:
    band_1       (zone_id) float32 11MB 72.2 72.13 72.2 ... 68.02 68.35 68.02

In [6]:
# Apply mean aggregation to the dataset by zone ID
igeo7_indexed_ds = igeo7_indexed_ds.groupby('zone_id').mean()

## Accessing IGEO7 indexed DGGS xarray datasets using `xdggs-dggrid4py`

After the conversion is done, the dataset is indexed with IGEO7 DGGS. However, the index is not meaningful and cannot be used by others. Therefore, `xdggs-dggrid4py` provides the `IGEO7Index` as an interface between the user and the zone ID. It is implemented as a plugin of [xdggs](https://github.com/xarray-contrib/xdggs/tree/main).

In [7]:
# import xdggs and register IGEO7Index by importing the IGEO7Index
import xdggs
from xdggs_dggrid4py.index import IGEO7Index

In [8]:
# since the xdggs convention uses the name `cell_ids` as the DGGS zone ID, we have to rename the `zone_id` to `cell_ids`
igeo7_indexed_ds = igeo7_indexed_ds.rename({'zone_id': 'cell_ids'})

### Create a IGEO7Index instance for the IGEO7 DGGS dataset
- using the xdggs.decode function to create an IGEO7Index index object for the dataset
- From the output, the `decode` function created an IGEO7Index object using the attributes from `cell_ids` (zone_id)

In [9]:
igeo7_indexed_ds = igeo7_indexed_ds.pipe(xdggs.decode)
igeo7_indexed_ds

<xarray.Dataset> Size: 31MB
Dimensions:      (cell_ids: 2581491)
Coordinates:
  * cell_ids     (cell_ids) uint64 21MB 18667837283631103 ... 18805349850808319
    spatial_ref  int64 8B 0
Data variables:
    band_1       (cell_ids) float32 10MB 72.2 72.13 72.2 ... 68.02 68.35 68.02
Indexes:
    cell_ids  IGEO7Index(level=14)

### Using IGEO7Index to access the dataset
- xdggs provides several accessor functions through `<dataset>.dggs`

In [10]:
lats = [58.28459946, 58.27637829, 58.26980135]
lons = [26.41758320, 26.42799669,  26.36551576]
igeo7_indexed_subset = igeo7_indexed_ds.dggs.sel_latlon(lats, lons)
igeo7_indexed_subset

<xarray.Dataset> Size: 44B
Dimensions:      (cell_ids: 3)
Coordinates:
  * cell_ids     (cell_ids) uint64 24B 18802036072775679 ... 18801924489347071
    spatial_ref  int64 8B 0
Data variables:
    band_1       (cell_ids) float32 12B 48.35 51.44 83.53
Indexes:
    cell_ids  IGEO7Index(level=14)

In [11]:
d = igeo7_indexed_subset['band_1']

In [12]:
d.dggs.explore()

ValueError: Expected object with __arrow_c_array__ or __arrow_c_stream__ method or implementing buffer protocol.